# Emergency Vehicle Detection — YOLO Model Training

**Project:** Emergency Vehicle Detection and Adaptive Signal Preemption for Heterogeneous Traffic Conditions

**8 Classes:** car, motorcycle, autorickshaw, truck, bus, bicycle, ambulance, police_car

**Dataset:** 9,739 images (6,836 train / 1,412 val / 1,491 test)

**Model:** YOLOv8n (nano) — lightweight for CPU inference on laptop

---
## Instructions
1. Upload your `split_dataset.zip` to Google Drive
2. Set `Runtime → Change runtime type → T4 GPU`
3. Run all cells in order
4. Download `best.pt` at the end

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Install ultralytics
!pip install ultralytics>=8.0.0 -q

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")
ultralytics.checks()

## 2. Mount Google Drive & Prepare Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile
from pathlib import Path

# === CONFIGURE THESE PATHS ===
# Option A: Upload zip to Google Drive and unzip
DRIVE_ZIP = '/content/drive/MyDrive/split_dataset.zip'  # <-- EDIT this path
DATASET_DIR = '/content/split_dataset'

# Option B: If dataset is already unzipped on Drive
# DATASET_DIR = '/content/drive/MyDrive/split_dataset'

# Unzip if needed
if os.path.exists(DRIVE_ZIP) and not os.path.exists(DATASET_DIR):
    print(f"Extracting {DRIVE_ZIP}...")
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall('/content/')
    print("Done.")
elif os.path.exists(DATASET_DIR):
    print(f"Dataset already exists at {DATASET_DIR}")
else:
    print(f"WARNING: Neither {DRIVE_ZIP} nor {DATASET_DIR} found!")
    print("Upload your dataset zip to Google Drive first.")

# Verify structure
for split in ['train', 'val', 'test']:
    img_dir = Path(DATASET_DIR) / split / 'images'
    lbl_dir = Path(DATASET_DIR) / split / 'labels'
    if img_dir.exists() and lbl_dir.exists():
        imgs = len(list(img_dir.glob('*')))
        lbls = len(list(lbl_dir.glob('*.txt')))
        print(f"{split}: {imgs} images, {lbls} labels")
    else:
        print(f"{split}: DIRECTORY NOT FOUND")

In [ ]:
# Create data.yaml for Colab paths
DATA_YAML = '/content/data.yaml'

yaml_content = f"""# Emergency Vehicle Detection Dataset (Colab)
path: {DATASET_DIR}

train: train/images
val: val/images
test: test/images

nc: 8

names:
  0: car
  1: motorcycle
  2: autorickshaw
  3: truck
  4: bus
  5: bicycle
  6: ambulance
  7: police_car
"""

with open(DATA_YAML, 'w') as f:
    f.write(yaml_content)

print(f"Data YAML saved: {DATA_YAML}")
print(open(DATA_YAML).read())

## 3. Quick Dataset Validation

In [ ]:
from collections import Counter
from pathlib import Path

CLASS_NAMES = {0:'car',1:'motorcycle',2:'autorickshaw',3:'truck',4:'bus',5:'bicycle',6:'ambulance',7:'police_car'}

for split in ['train', 'val', 'test']:
    lbl_dir = Path(DATASET_DIR) / split / 'labels'
    counts = Counter()
    total_boxes = 0
    for lbl_file in lbl_dir.glob('*.txt'):
        with open(lbl_file) as f:
            for line in f:
                line = line.strip()
                if line:
                    cid = int(line.split()[0])
                    counts[cid] += 1
                    total_boxes += 1
    print(f"\n{split} ({total_boxes} boxes):")
    for cid in range(8):
        tag = ' *** EMERGENCY' if cid in {6,7} else ''
        print(f"  {CLASS_NAMES[cid]:15s}: {counts[cid]:>6}{tag}")

## 4. Model Training

**Model:** YOLOv8n (nano) — 3.2M parameters, fast CPU inference

**Key choices:**
- `imgsz=640`: Standard YOLO input size, good balance of accuracy and speed
- `batch=16`: Fits T4 GPU memory (16GB) comfortably
- `epochs=100`: With patience=20, training will stop early if no improvement
- `patience=20`: Early stopping on val mAP
- `seed=42`: Reproducible
- `mosaic=1.0, mixup=0.1`: Standard augmentation for small object detection
- `cos_lr=True`: Cosine LR schedule for better convergence

In [ ]:
from ultralytics import YOLO
import time

# Output directories
PROJECT_DIR = '/content/training_output'
RUN_NAME = 'emergency_vehicle_v1'
DRIVE_SAVE_DIR = '/content/drive/MyDrive/emergency_vehicle_training'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Load pretrained YOLOv8n
model = YOLO('yolov8n.pt')
print(f"Model loaded: yolov8n.pt")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

# Train
start_time = time.time()

results = model.train(
    data=DATA_YAML,
    epochs=100,
    batch=16,
    imgsz=640,
    patience=20,
    seed=42,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    verbose=True,
    save=True,
    plots=True,
    val=True,
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    # LR schedule
    cos_lr=True,
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,
)

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed/60:.1f} minutes")

In [ ]:
# Save best weights to Google Drive immediately
import shutil

best_pt = Path(PROJECT_DIR) / RUN_NAME / 'weights' / 'best.pt'
last_pt = Path(PROJECT_DIR) / RUN_NAME / 'weights' / 'last.pt'

if best_pt.exists():
    shutil.copy2(best_pt, Path(DRIVE_SAVE_DIR) / 'best.pt')
    print(f"best.pt saved to Drive: {DRIVE_SAVE_DIR}/best.pt")
    print(f"File size: {best_pt.stat().st_size / 1e6:.1f} MB")

if last_pt.exists():
    shutil.copy2(last_pt, Path(DRIVE_SAVE_DIR) / 'last.pt')
    print(f"last.pt saved to Drive: {DRIVE_SAVE_DIR}/last.pt")

# Copy training plots
results_dir = Path(PROJECT_DIR) / RUN_NAME
for plot_file in results_dir.glob('*.png'):
    shutil.copy2(plot_file, Path(DRIVE_SAVE_DIR) / plot_file.name)
for csv_file in results_dir.glob('*.csv'):
    shutil.copy2(csv_file, Path(DRIVE_SAVE_DIR) / csv_file.name)
print("Training artifacts saved to Drive.")

## 5. Model Evaluation

In [ ]:
# Evaluate on VALIDATION set
best_model = YOLO(str(best_pt))

val_metrics = best_model.val(
    data=DATA_YAML,
    split='val',
    imgsz=640,
    verbose=True,
    plots=True,
)

print("\n" + "="*60)
print("VALIDATION RESULTS")
print("="*60)
print(f"mAP@50:     {val_metrics.box.map50:.4f}")
print(f"mAP@50-95:  {val_metrics.box.map:.4f}")

# Per-class metrics
names = best_model.names
print(f"\n{'Class':<15} {'Prec':>8} {'Recall':>8} {'AP@50':>8} {'AP@50-95':>10}")
print("-" * 55)
for i in range(8):
    p = val_metrics.box.p[i] if hasattr(val_metrics.box, 'p') else 0
    r = val_metrics.box.r[i] if hasattr(val_metrics.box, 'r') else 0
    ap50 = val_metrics.box.ap50[i] if hasattr(val_metrics.box, 'ap50') else 0
    ap = val_metrics.box.ap[i] if hasattr(val_metrics.box, 'ap') else 0
    tag = ' ***' if i in {6,7} else ''
    print(f"{names[i]:<15} {p:>8.4f} {r:>8.4f} {ap50:>8.4f} {ap:>10.4f}{tag}")

In [ ]:
# Evaluate on TEST set (final evaluation only — do NOT use for tuning)
test_metrics = best_model.val(
    data=DATA_YAML,
    split='test',
    imgsz=640,
    verbose=True,
    plots=True,
)

print("\n" + "="*60)
print("TEST RESULTS (FINAL — DO NOT USE FOR TUNING)")
print("="*60)
print(f"mAP@50:     {test_metrics.box.map50:.4f}")
print(f"mAP@50-95:  {test_metrics.box.map:.4f}")

print(f"\n{'Class':<15} {'Prec':>8} {'Recall':>8} {'AP@50':>8} {'AP@50-95':>10}")
print("-" * 55)
for i in range(8):
    p = test_metrics.box.p[i] if hasattr(test_metrics.box, 'p') else 0
    r = test_metrics.box.r[i] if hasattr(test_metrics.box, 'r') else 0
    ap50 = test_metrics.box.ap50[i] if hasattr(test_metrics.box, 'ap50') else 0
    ap = test_metrics.box.ap[i] if hasattr(test_metrics.box, 'ap') else 0
    tag = ' ***' if i in {6,7} else ''
    print(f"{names[i]:<15} {p:>8.4f} {r:>8.4f} {ap50:>8.4f} {ap:>10.4f}{tag}")

## 6. Sample Predictions

In [ ]:
from pathlib import Path
import random
from IPython.display import Image, display

# Run predictions on sample test images
test_imgs = list((Path(DATASET_DIR) / 'test' / 'images').glob('*.png'))
random.seed(42)
samples = random.sample(test_imgs, min(8, len(test_imgs)))

pred_results = best_model.predict(
    source=samples,
    imgsz=640,
    conf=0.25,
    save=True,
    project=PROJECT_DIR,
    name='sample_predictions',
    exist_ok=True,
)

# Display results
pred_dir = Path(PROJECT_DIR) / 'sample_predictions'
for img_file in sorted(pred_dir.glob('*.png'))[:8]:
    print(f"\n--- {img_file.name} ---")
    display(Image(filename=str(img_file), width=500))

In [ ]:
# Find and predict on emergency vehicle images specifically
emergency_test_imgs = []
test_lbl_dir = Path(DATASET_DIR) / 'test' / 'labels'
test_img_dir = Path(DATASET_DIR) / 'test' / 'images'

for lbl_file in test_lbl_dir.glob('*.txt'):
    with open(lbl_file) as f:
        content = f.read()
    # Check for ambulance (6) or police_car (7)
    for line in content.strip().split('\n'):
        if line.strip():
            cid = int(line.split()[0])
            if cid in {6, 7}:
                img_path = test_img_dir / (lbl_file.stem + '.png')
                if img_path.exists():
                    emergency_test_imgs.append((img_path, 'ambulance' if cid == 6 else 'police_car'))
                break

print(f"Emergency test images found: {len(emergency_test_imgs)}")

# Sample and predict
random.seed(42)
emg_samples = random.sample(emergency_test_imgs, min(6, len(emergency_test_imgs)))

for img_path, cls_name in emg_samples:
    result = best_model.predict(
        source=str(img_path),
        imgsz=640,
        conf=0.25,
        save=True,
        project=PROJECT_DIR,
        name='emergency_predictions',
        exist_ok=True,
    )
    # Check if emergency was detected
    if result and result[0].boxes is not None:
        detected_classes = result[0].boxes.cls.cpu().numpy().tolist()
        detected_names = [names[int(c)] for c in detected_classes]
        confs = result[0].boxes.conf.cpu().numpy().tolist()
        emg_detected = any(int(c) in {6,7} for c in detected_classes)
        print(f"\n{img_path.name} (GT: {cls_name})")
        print(f"  Detected: {detected_names}")
        print(f"  Confs: {[f'{c:.2f}' for c in confs]}")
        print(f"  Emergency detected: {'YES ✓' if emg_detected else 'NO ✗'}")

# Display emergency predictions
emg_pred_dir = Path(PROJECT_DIR) / 'emergency_predictions'
for img_file in sorted(emg_pred_dir.glob('*.png'))[:6]:
    display(Image(filename=str(img_file), width=500))

## 7. Save Everything to Google Drive

In [ ]:
import shutil

# Save all training artifacts
results_dir = Path(PROJECT_DIR) / RUN_NAME

# Copy everything to Drive
drive_results = Path(DRIVE_SAVE_DIR) / 'results'
if drive_results.exists():
    shutil.rmtree(drive_results)
shutil.copytree(results_dir, drive_results)

# Copy prediction samples
for pred_name in ['sample_predictions', 'emergency_predictions']:
    pred_dir = Path(PROJECT_DIR) / pred_name
    if pred_dir.exists():
        drive_pred = Path(DRIVE_SAVE_DIR) / pred_name
        if drive_pred.exists():
            shutil.rmtree(drive_pred)
        shutil.copytree(pred_dir, drive_pred)

print(f"All artifacts saved to: {DRIVE_SAVE_DIR}")
print("\nFiles saved:")
for f in sorted(Path(DRIVE_SAVE_DIR).rglob('*')):
    if f.is_file():
        print(f"  {f.relative_to(DRIVE_SAVE_DIR)} ({f.stat().st_size/1e6:.1f} MB)")

## 8. Download best.pt

Download `best.pt` and place it at:
```
Emergency_Vehicle_Project/models/best.pt
```

In [ ]:
from google.colab import files

best_pt_path = Path(PROJECT_DIR) / RUN_NAME / 'weights' / 'best.pt'
if best_pt_path.exists():
    print(f"Downloading best.pt ({best_pt_path.stat().st_size/1e6:.1f} MB)...")
    files.download(str(best_pt_path))
else:
    print("ERROR: best.pt not found!")
    # Try Drive location
    drive_best = Path(DRIVE_SAVE_DIR) / 'best.pt'
    if drive_best.exists():
        print(f"Found on Drive: {drive_best}")
        files.download(str(drive_best))

## 9. Training Summary Report

In [ ]:
# Print final summary
print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Model: YOLOv8n")
print(f"Dataset: 6,836 train / 1,412 val / 1,491 test")
print(f"Image size: 640")
print(f"Training time: {elapsed/60:.1f} minutes")
print(f"\nValidation mAP@50: {val_metrics.box.map50:.4f}")
print(f"Validation mAP@50-95: {val_metrics.box.map:.4f}")
print(f"\nTest mAP@50: {test_metrics.box.map50:.4f}")
print(f"Test mAP@50-95: {test_metrics.box.map:.4f}")
print(f"\nModel saved to:")
print(f"  Colab: {best_pt}")
print(f"  Drive: {DRIVE_SAVE_DIR}/best.pt")
print(f"\nNext steps:")
print(f"  1. Download best.pt")
print(f"  2. Place in Emergency_Vehicle_Project/models/best.pt")
print(f"  3. Test with: .venv\\Scripts\\python.exe -c \"from ultralytics import YOLO; m=YOLO('models/best.pt'); print(m.names)\"")
print(f"  4. Run app: .venv\\Scripts\\streamlit.exe run app.py")